# Chapitre 9 - Cours 3 : PRIM, MARS, mélanges d'experts et données manquantes

Ce dernier notebook de cours couvre les sections 9.3 à 9.7 du livre :

1. **PRIM** (*patient rule induction method*), la recherche de « bosses » par
   épluchage de boîtes (algorithme 9.3, figures 9.7 et 9.8, exemple spam) ;
2. **MARS** (*multivariate adaptive regression splines*) : fonctions charnières,
   passe avant, passe arrière avec GCV, scénarios simulés (table 9.4), lien avec CART ;
3. les **mélanges hiérarchiques d'experts** (HME), version « douce » des arbres, avec un
   petit ajustement par l'algorithme EM ;
4. les **données manquantes** : mécanismes MCAR / MAR, imputation, coupures de
   substitution des arbres ;
5. les coûts de calcul des différentes méthodes.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from esl.plotting import set_style, COLORS, BLUE, ORANGE, GRAY, SEQ, DIV, TWO_CLASS
from esl import datasets
from esl.ch09_arbres import (prim_peel, prim_paste, prim, in_box, box_rules, MARS,
                             ArbreCART, SPAM_NAMES)

set_style()
rng = np.random.RandomState(0)

## 1. PRIM : la chasse aux bosses

Les arbres de régression cherchent des boîtes dont les moyennes de réponse sont **aussi
différentes que possible**. PRIM cherche plutôt des boîtes où la moyenne de la réponse
est **élevée** : il vise les **maxima** de la fonction cible (*bump hunting*). Pour
chercher des minima, il suffit de changer $y$ en $-y$.

Autre différence : les boîtes de PRIM ne sont **pas** organisées en arbre binaire. Leur
ensemble est un peu moins facile à interpréter, mais chaque règle est souvent plus
simple.

### L'algorithme 9.3

1. Partir de toutes les données et de la boîte maximale qui les contient.
2. **Épluchage** : envisager de comprimer la boîte sur une de ses faces, de façon à
   retirer la fraction $\alpha$ des observations ayant les plus grandes (ou les plus
   petites) valeurs d'une variable $X_j$. Retenir l'épluchage qui donne la plus grande
   moyenne de réponse dans la boîte restante ($\alpha = 0.05$ ou $0.10$ typiquement).
3. Répéter l'étape 2 jusqu'à ce qu'il reste un nombre minimal d'observations (10, par
   exemple).
4. **Recollage** : élargir la boîte le long d'une face tant que cela augmente la
   moyenne de la boîte.
5. Ces étapes donnent une suite de boîtes d'effectifs décroissants ; on en choisit une
   par validation croisée (et jugement de l'analyste) : c'est $B_1$.
6. Retirer les observations de $B_1$ et recommencer pour obtenir $B_2$, $B_3$, etc.